# INST414 — Lab 10: Repeated Modeling Workflows with Gradient Boosting

This lab uses data from a speed-dating experiment. Participants went on a series of short dates. After each date, each person decided whether they wanted to see the other person again.

Each row is one dater evaluating one partner. The same real-world date may appear twice: once from each person's perspective.

The prediction task is:

`P(decision = 1 | dater features, partner features, ratings, preferences)`

where `decision = 1` means this dater said yes to seeing this partner again.

This is an end-of-date prediction task. Some features are ratings collected after the date, so this is not a pre-date prediction model.

The main coding goal is to organize a workflow that can be repeated while changing the feature set and model settings.

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

pd.options.display.max_columns = None

## Load The Data

The original file is from OpenML's `SpeedDating` dataset. For this lab, the ARFF file has been converted to CSV so it is easier to load with pandas.

In [ ]:
speed = pd.read_csv(
    filepath_or_buffer='https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/master/files/inst414_spring2026/labs/data/speeddating.csv'
)
speed.shape

In [ ]:
speed.head()

## Unit Of Analysis And Outcome

Each row is one dater evaluating one partner.

Important columns:

- `decision`: whether this dater said yes
- `decision_o`: whether the partner said yes
- `match`: whether both people said yes
- `gender`: gender of this dater
- `age`, `race`, preferences, and self-ratings: attributes of this dater
- `age_o`, `race_o`, and variables ending in `_o`: attributes or ratings from the partner side
- variables ending in `_partner`: this dater's rating of the partner

This lab predicts `decision`, not `match`. The `match` outcome is symmetric because a mutual match is shared by both sides of the pair. The `decision` outcome is directional: it is this dater's choice.

In [ ]:
speed[['decision', 'decision_o', 'match', 'gender', 'age', 'age_o', 'd_age']].tail()

In [ ]:
# Base rate: share of rows where the dater said yes.
speed['decision'].mean()

In [ ]:
# Decision rates by dater gender.
speed.groupby('gender')['decision'].mean()

## Feature Sets

The feature sets build up by adding groups of related features.

The first feature set uses basic date context plus intelligence-related variables.

The second feature set adds funny-related variables.

The third feature set adds attractiveness-related variables.

The fourth feature set adds `gender_male`, which lets the model use the dater's gender directly.

Sincerity-related variables are prepared but held out for the lab tasks.

Each trait group includes three kinds of information when available:

- how the dater rates themself
- how important that trait is to the dater
- how the dater rates the partner on that trait

`interests_correlate` measures how similar the dater's and partner's interest profiles are. It is a compatibility measure based on ratings of interests such as sports, dining, museums, music, shopping, and yoga.

Do not use `decision_o` or `match` as features. Those variables are too close to the outcome.

In [ ]:
basic_features = [
    'age',
    'age_o',
    'd_age',
    'interests_correlate'
]

intelligence_features = [
    'intelligence',
    'intellicence_important',
    'intelligence_partner'
]

funny_features = [
    'funny',
    'funny_important',
    'funny_partner'
]

attractiveness_features = [
    'attractive',
    'attractive_important',
    'attractive_partner'
]

sincerity_features = [
    'sincere',
    'sincere_important',
    'sincere_partner'
]

## Prepare A Modeling Dataset

For this lab, missing values in the selected feature columns are filled with `0`. This keeps all rows in the dataset and keeps the focus on the repeated modeling workflow. In a real project, the missing-data strategy would need more attention.

In [ ]:
selected_columns = (
    ['decision', 'gender']
    + basic_features
    + intelligence_features
    + funny_features
    + attractiveness_features
    + sincerity_features
)

model_data = speed[selected_columns].copy()

# Create a numeric gender variable using pandas.
gender_dummies = pd.get_dummies(model_data['gender'])
model_data['gender_male'] = gender_dummies['male']

# Fill missing feature values with 0 for this lab.
model_data = model_data.fillna(0)

model_data.head()

In [ ]:
model_data.shape

## Create One Train/Test Split

Use one train/test split and reuse it for all model specifications. This keeps the model comparisons clean because each model is evaluated on the same test cases.

In [ ]:
train, test = train_test_split(
    model_data,
    test_size=0.30,
    random_state=414,
    stratify=model_data['decision']
)

train.shape, test.shape

## Fit One Gradient Boosting Model

Start with one model before abstracting the workflow. The first model predicts `decision` using basic features plus intelligence-related features.

In [ ]:
first_features = basic_features + intelligence_features

first_model = GradientBoostingClassifier(
    learning_rate=0.10,
    max_depth=2,
    n_estimators=100,
    random_state=414
)

first_model.fit(X=train[first_features], y=train['decision'])

test = test.copy()
test['pred_first_model'] = first_model.predict_proba(X=test[first_features])[:, 1]

roc_auc_score(y_true=test['decision'], y_score=test['pred_first_model'])

In [ ]:
# Look at predicted probabilities for cases where the dater actually said yes.
sns.histplot(
    data=test[test['decision'] == 1],
    x='pred_first_model',
    bins=30
)
plt.title('Predicted Probabilities When Decision = 1')
plt.xlabel('Predicted probability of saying yes')
plt.ylabel('Count')
plt.tight_layout()
plt.show();

## Abstract The Repeated Parts

When a workflow repeats, create objects that describe what changes and objects that store what comes out.

Objects used here:

- `feature_sets`: named feature lists
- `model_specs`: one row of instructions per model to fit
- `best_models`: fitted models stored by name
- `model_results`: summary rows with best settings and test performance
- `predictions`: test set plus predicted probabilities

This pattern is useful because model fitting, prediction, and evaluation all need the same labels and feature lists.

In [ ]:
feature_sets = {
    'basic_plus_intelligence': basic_features + intelligence_features,
    'basic_plus_intelligence_funny': basic_features + intelligence_features + funny_features,
    'basic_plus_intelligence_funny_attractiveness': basic_features + intelligence_features + funny_features + attractiveness_features,
    'basic_plus_intelligence_funny_attractiveness_gender': basic_features + intelligence_features + funny_features + attractiveness_features + ['gender_male']
}

model_specs = [
    {
        'model_label': 'basic_plus_intelligence',
        'features': feature_sets['basic_plus_intelligence']
    },
    {
        'model_label': 'basic_plus_intelligence_funny',
        'features': feature_sets['basic_plus_intelligence_funny']
    },
    {
        'model_label': 'basic_plus_intelligence_funny_attractiveness',
        'features': feature_sets['basic_plus_intelligence_funny_attractiveness']
    },
    {
        'model_label': 'basic_plus_intelligence_funny_attractiveness_gender',
        'features': feature_sets['basic_plus_intelligence_funny_attractiveness_gender']
    }
]

param_grid = {
    'learning_rate': [0.01, 0.05, 0.10],
    'max_depth': [1, 2, 3],
    'n_estimators': [10, 100, 200]
}

In [ ]:
# This is what the model specification object looks like.
model_specs

## Tune The Models

Each model specification gets the same tuning process:

1. Use `GridSearchCV` on the training set.
2. Tune `learning_rate`, `max_depth`, and `n_estimators`.
3. Store the best fitted model.
4. Add predicted probabilities to the shared test-set table.
5. Store a summary row.

In [ ]:
best_models = {}
model_results = []
predictions = test.copy()

for spec in model_specs:
    # Pull the model label and feature list out of the specification.
    model_label = spec['model_label']
    features = spec['features']

    # Tune learning_rate, max_depth, and n_estimators using cross-validation on the training set.
    grid_search = GridSearchCV(
        estimator=GradientBoostingClassifier(random_state=414),
        param_grid=param_grid,
        scoring='roc_auc',
        cv=5
    )

    # Fit only on the training set.
    grid_search.fit(X=train[features], y=train['decision'])

    # Store the best fitted model so it can be reused later.
    best_models[model_label] = grid_search.best_estimator_

    # Add test-set predicted probabilities to one shared predictions table.
    pred_col = 'pred_' + model_label
    predictions[pred_col] = grid_search.predict_proba(X=predictions[features])[:, 1]

    # Evaluate the chosen model on the test set.
    test_auc = roc_auc_score(
        y_true=predictions['decision'],
        y_score=predictions[pred_col]
    )

    # Store one row of results for this model.
    model_results.append({
        'model_label': model_label,
        'num_features': len(features),
        'best_learning_rate': grid_search.best_params_['learning_rate'],
        'best_max_depth': grid_search.best_params_['max_depth'],
        'best_n_estimators': grid_search.best_params_['n_estimators'],
        'test_auc': test_auc
    })

model_results = pd.DataFrame(model_results)
model_results

## Evaluate A Top-K Rule

Suppose the app can only show a limited number of high-probability potential dates. A top-k rule flags the highest-scoring rows.

Here, `k = 250`, so each model flags the 250 rows with the highest predicted probability.

In [ ]:
k = 250

topk_results = []

for spec in model_specs:
    model_label = spec['model_label']
    pred_col = 'pred_' + model_label
    flag_col = 'flag_top_' + model_label

    topk_index = predictions.sort_values(by=pred_col, ascending=False).head(k).index
    predictions[flag_col] = 0
    predictions.loc[topk_index, flag_col] = 1

    precision_at_k = predictions.loc[topk_index, 'decision'].mean()

    topk_results.append({
        'model_label': model_label,
        'k': k,
        'share_flagged': k / len(predictions),
        'precision_at_k': precision_at_k
    })

topk_results = pd.DataFrame(topk_results)
topk_results

## Fairness Analysis By Gender

Now compare the top-k flags by gender.

Demographic parity compares the rate at which groups receive the positive prediction. Here, the positive prediction is being flagged in the top 250.

For each model, compute:

- gender-specific AUC
- average predicted probability among flagged rows
- flag rate: `P(flagged = 1 | gender)`
- PPV: among flagged rows, the share where `decision = 1`
- FPR: among rows where `decision = 0`, the share flagged
- FNR: among rows where `decision = 1`, the share not flagged

In [ ]:
fairness_rows = []

for spec in model_specs:
    model_label = spec['model_label']
    pred_col = 'pred_' + model_label
    flag_col = 'flag_top_' + model_label

    for gender in ['female', 'male']:
        group = predictions[predictions['gender'] == gender]

        tp = ((group[flag_col] == 1) & (group['decision'] == 1)).sum()
        fp = ((group[flag_col] == 1) & (group['decision'] == 0)).sum()
        fn = ((group[flag_col] == 0) & (group['decision'] == 1)).sum()
        tn = ((group[flag_col] == 0) & (group['decision'] == 0)).sum()

        gender_auc = roc_auc_score(
            y_true=group['decision'],
            y_score=group[pred_col]
        )

        fairness_rows.append({
            'model_label': model_label,
            'gender': gender,
            'AUC': gender_auc,
            'avg_pred_prob_flagged': group[group[flag_col] == 1][pred_col].mean(),
            'flag_rate': group[flag_col].mean(),
            'PPV': tp / (tp + fp),
            'FPR': fp / (fp + tn),
            'FNR': fn / (fn + tp)
        })

fairness_results = pd.DataFrame(fairness_rows)
fairness_results